In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, rank, dense_rank, lag, lead, sum, avg
from pyspark.sql.window import Window

In [3]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

In [4]:
spark = (
        SparkSession.builder
        .appName("AnalyticalFunctions App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
io.prometheus.jmx#jmx_prometheus_javaagent added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c49a0e85-704c-477d-b502-2f8304dfd2b8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in cent

In [5]:
spark

26/04/29 14:15:40 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
data = [
    (1, "Alice",   "HR",       3500, "2022-01-10"),
    (2, "Bob",     "HR",       4200, "2022-01-15"),
    (3, "Charlie", "IT",       6000, "2022-02-01"),
    (4, "David",   "IT",       7200, "2022-02-10"),
    (5, "Eve",     "Finance",  5100, "2022-03-05"),
    (6, "Frank",   "Finance",  6200, "2022-03-20"),
    (7, "Grace",   "IT",       5500, "2022-04-01"),
    (8, "Hank",    "HR",       3000, "2022-04-12"),
    (9, "Ivy",     "Finance",  7000, "2022-05-02"),
    (10,"Jack",    "IT",       8000, "2022-05-15")
]
columns = ["emp_id", "name", "dept", "salary", "hire_date"]
df = spark.createDataFrame(data, columns)

In [7]:
df.show()

+------+-------+-------+------+----------+
|emp_id|   name|   dept|salary| hire_date|
+------+-------+-------+------+----------+
|     1|  Alice|     HR|  3500|2022-01-10|
|     2|    Bob|     HR|  4200|2022-01-15|
|     3|Charlie|     IT|  6000|2022-02-01|
|     4|  David|     IT|  7200|2022-02-10|
|     5|    Eve|Finance|  5100|2022-03-05|
|     6|  Frank|Finance|  6200|2022-03-20|
|     7|  Grace|     IT|  5500|2022-04-01|
|     8|   Hank|     HR|  3000|2022-04-12|
|     9|    Ivy|Finance|  7000|2022-05-02|
|    10|   Jack|     IT|  8000|2022-05-15|
+------+-------+-------+------+----------+



#### row_number, rank, dense_rank

In [6]:
windowSpec = Window.partitionBy("dept").orderBy(col("salary").desc())

df.withColumn("row_number", row_number().over(windowSpec)) \
  .withColumn("rank", rank().over(windowSpec)) \
  .withColumn("dense_rank", dense_rank().over(windowSpec)) \
  .show()

+------+-------+-------+------+----------+----------+----+----------+
|emp_id|   name|   dept|salary| hire_date|row_number|rank|dense_rank|
+------+-------+-------+------+----------+----------+----+----------+
|     9|    Ivy|Finance|  7000|2022-05-02|         1|   1|         1|
|     6|  Frank|Finance|  6200|2022-03-20|         2|   2|         2|
|     5|    Eve|Finance|  5100|2022-03-05|         3|   3|         3|
|     2|    Bob|     HR|  4200|2022-01-15|         1|   1|         1|
|     1|  Alice|     HR|  3500|2022-01-10|         2|   2|         2|
|     8|   Hank|     HR|  3000|2022-04-12|         3|   3|         3|
|    10|   Jack|     IT|  8000|2022-05-15|         1|   1|         1|
|     4|  David|     IT|  7200|2022-02-10|         2|   2|         2|
|     3|Charlie|     IT|  6000|2022-02-01|         3|   3|         3|
|     7|  Grace|     IT|  5500|2022-04-01|         4|   4|         4|
+------+-------+-------+------+----------+----------+----+----------+



In [7]:
windowSpec = Window.partitionBy("dept").orderBy(col("salary").asc())

df.withColumn("row_number", row_number().over(windowSpec)) \
  .withColumn("rank", rank().over(windowSpec)) \
  .withColumn("dense_rank", dense_rank().over(windowSpec)) \
  .show()

+------+-------+-------+------+----------+----------+----+----------+
|emp_id|   name|   dept|salary| hire_date|row_number|rank|dense_rank|
+------+-------+-------+------+----------+----------+----+----------+
|     5|    Eve|Finance|  5100|2022-03-05|         1|   1|         1|
|     6|  Frank|Finance|  6200|2022-03-20|         2|   2|         2|
|     9|    Ivy|Finance|  7000|2022-05-02|         3|   3|         3|
|     8|   Hank|     HR|  3000|2022-04-12|         1|   1|         1|
|     1|  Alice|     HR|  3500|2022-01-10|         2|   2|         2|
|     2|    Bob|     HR|  4200|2022-01-15|         3|   3|         3|
|     7|  Grace|     IT|  5500|2022-04-01|         1|   1|         1|
|     3|Charlie|     IT|  6000|2022-02-01|         2|   2|         2|
|     4|  David|     IT|  7200|2022-02-10|         3|   3|         3|
|    10|   Jack|     IT|  8000|2022-05-15|         4|   4|         4|
+------+-------+-------+------+----------+----------+----+----------+



#### Lead & Lag

In [8]:
df.withColumn("next_salary", lead("salary", 1).over(windowSpec)) \
  .withColumn("prev_salary", lag("salary", 1).over(windowSpec)) \
  .withColumn("2_next_salary", lead("salary", 2).over(windowSpec)) \
  .withColumn("2_prev_salary", lag("salary", 2).over(windowSpec)) \
  .show()

+------+-------+-------+------+----------+-----------+-----------+-------------+-------------+
|emp_id|   name|   dept|salary| hire_date|next_salary|prev_salary|2_next_salary|2_prev_salary|
+------+-------+-------+------+----------+-----------+-----------+-------------+-------------+
|     5|    Eve|Finance|  5100|2022-03-05|       6200|       NULL|         7000|         NULL|
|     6|  Frank|Finance|  6200|2022-03-20|       7000|       5100|         NULL|         NULL|
|     9|    Ivy|Finance|  7000|2022-05-02|       NULL|       6200|         NULL|         5100|
|     8|   Hank|     HR|  3000|2022-04-12|       3500|       NULL|         4200|         NULL|
|     1|  Alice|     HR|  3500|2022-01-10|       4200|       3000|         NULL|         NULL|
|     2|    Bob|     HR|  4200|2022-01-15|       NULL|       3500|         NULL|         3000|
|     7|  Grace|     IT|  5500|2022-04-01|       6000|       NULL|         7200|         NULL|
|     3|Charlie|     IT|  6000|2022-02-01|       7

#### Aggregates with Window

In [9]:
aggSpec = Window.partitionBy("dept")

df.withColumn("dept_total", sum("salary").over(aggSpec)) \
  .withColumn("dept_avg", avg("salary").over(aggSpec)) \
  .withColumn("running_total", sum("salary").over(windowSpec.rowsBetween(Window.unboundedPreceding, 0))) \
  .show()

+------+-------+-------+------+----------+----------+------------------+-------------+
|emp_id|   name|   dept|salary| hire_date|dept_total|          dept_avg|running_total|
+------+-------+-------+------+----------+----------+------------------+-------------+
|     5|    Eve|Finance|  5100|2022-03-05|     18300|            6100.0|         5100|
|     6|  Frank|Finance|  6200|2022-03-20|     18300|            6100.0|        11300|
|     9|    Ivy|Finance|  7000|2022-05-02|     18300|            6100.0|        18300|
|     8|   Hank|     HR|  3000|2022-04-12|     10700|3566.6666666666665|         3000|
|     1|  Alice|     HR|  3500|2022-01-10|     10700|3566.6666666666665|         6500|
|     2|    Bob|     HR|  4200|2022-01-15|     10700|3566.6666666666665|        10700|
|     7|  Grace|     IT|  5500|2022-04-01|     26700|            6675.0|         5500|
|     3|Charlie|     IT|  6000|2022-02-01|     26700|            6675.0|        11500|
|     4|  David|     IT|  7200|2022-02-10| 

### New DataFrame

In [10]:
data = [
    (1, "Alice", "HR", "New York", 3500, "2022-01-10"),
    (2, "Bob", "HR", "New York", 4200, "2022-01-15"),
    (3, "Charlie", "IT", "Chicago", 6000, "2022-02-01"),
    (4, "David", "IT", "Chicago", 7200, "2022-02-10"),
    (5, "Eve", "Finance", "New York", 5100, "2022-03-05"),
    (6, "Frank", "Finance", "Boston", 6200, "2022-03-20"),
    (7, "Grace", "IT", "Boston", 5500, "2022-04-01"),
    (8, "Hank", "HR", "Boston", 3000, "2022-04-12"),
    (9, "Ivy", "Finance", "Chicago", 7000, "2022-05-02"),
    (10,"Jack", "IT", "Boston", 8000, "2022-05-15"),
    (11,"Ken", "Finance", "New York", 4800, "2022-06-01"),
    (12,"Lily", "HR", "Chicago", 3900, "2022-06-10")
]

columns = ["emp_id", "name", "dept", "location", "salary", "hire_date"]

df = spark.createDataFrame(data, columns)
df.show()

+------+-------+-------+--------+------+----------+
|emp_id|   name|   dept|location|salary| hire_date|
+------+-------+-------+--------+------+----------+
|     1|  Alice|     HR|New York|  3500|2022-01-10|
|     2|    Bob|     HR|New York|  4200|2022-01-15|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|
|     4|  David|     IT| Chicago|  7200|2022-02-10|
|     5|    Eve|Finance|New York|  5100|2022-03-05|
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|
|    10|   Jack|     IT|  Boston|  8000|2022-05-15|
|    11|    Ken|Finance|New York|  4800|2022-06-01|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|
+------+-------+-------+--------+------+----------+



In [11]:
win = Window.partitionBy("location", "dept").orderBy(col("salary").asc())

In [12]:
df.withColumn("row_number", row_number().over(win)) \
    .withColumn("rank", rank().over(win)) \
    .withColumn("dense_rank", dense_rank().over(win)).show()

+------+-------+-------+--------+------+----------+----------+----+----------+
|emp_id|   name|   dept|location|salary| hire_date|row_number|rank|dense_rank|
+------+-------+-------+--------+------+----------+----------+----+----------+
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|         1|   1|         1|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|         1|   1|         1|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|         1|   1|         1|
|    10|   Jack|     IT|  Boston|  8000|2022-05-15|         2|   2|         2|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|         1|   1|         1|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|         1|   1|         1|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|         1|   1|         1|
|     4|  David|     IT| Chicago|  7200|2022-02-10|         2|   2|         2|
|    11|    Ken|Finance|New York|  4800|2022-06-01|         1|   1|         1|
|     5|    Eve|Finance|New York|  5100|2022-03-05| 

In [13]:
df.withColumn("lead", lead("salary", 1).over(Window.partitionBy("dept").orderBy(col("salary").asc()))).withColumn("lag", lag("salary", 1).over(win)).show()

+------+-------+-------+--------+------+----------+----+----+
|emp_id|   name|   dept|location|salary| hire_date|lead| lag|
+------+-------+-------+--------+------+----------+----+----+
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|7000|NULL|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|3500|NULL|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|6000|NULL|
|    10|   Jack|     IT|  Boston|  8000|2022-05-15|NULL|5500|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|NULL|NULL|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|4200|NULL|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|7200|NULL|
|     4|  David|     IT| Chicago|  7200|2022-02-10|8000|6000|
|    11|    Ken|Finance|New York|  4800|2022-06-01|5100|NULL|
|     5|    Eve|Finance|New York|  5100|2022-03-05|6200|4800|
|     1|  Alice|     HR|New York|  3500|2022-01-10|3900|NULL|
|     2|    Bob|     HR|New York|  4200|2022-01-15|NULL|3500|
+------+-------+-------+--------+------+----------+----+----+



#### rowsBetween

In [14]:
df.show()

+------+-------+-------+--------+------+----------+
|emp_id|   name|   dept|location|salary| hire_date|
+------+-------+-------+--------+------+----------+
|     1|  Alice|     HR|New York|  3500|2022-01-10|
|     2|    Bob|     HR|New York|  4200|2022-01-15|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|
|     4|  David|     IT| Chicago|  7200|2022-02-10|
|     5|    Eve|Finance|New York|  5100|2022-03-05|
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|
|    10|   Jack|     IT|  Boston|  8000|2022-05-15|
|    11|    Ken|Finance|New York|  4800|2022-06-01|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|
+------+-------+-------+--------+------+----------+



In [15]:
win = Window.partitionBy("dept").orderBy("salary")

In [16]:
df.withColumn("Running Salary Total", sum("salary").over(win.rowsBetween(Window.unboundedPreceding, 0))).show()

+------+-------+-------+--------+------+----------+--------------------+
|emp_id|   name|   dept|location|salary| hire_date|Running Salary Total|
+------+-------+-------+--------+------+----------+--------------------+
|    11|    Ken|Finance|New York|  4800|2022-06-01|                4800|
|     5|    Eve|Finance|New York|  5100|2022-03-05|                9900|
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|               16100|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|               23100|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|                3000|
|     1|  Alice|     HR|New York|  3500|2022-01-10|                6500|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|               10400|
|     2|    Bob|     HR|New York|  4200|2022-01-15|               14600|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|                5500|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|               11500|
|     4|  David|     IT| Chicago|  7200|2022-02-10|

In [17]:
df.withColumn("Running Salary Total", sum("salary").over(win.rowsBetween(-1, 1))).show()

+------+-------+-------+--------+------+----------+--------------------+
|emp_id|   name|   dept|location|salary| hire_date|Running Salary Total|
+------+-------+-------+--------+------+----------+--------------------+
|    11|    Ken|Finance|New York|  4800|2022-06-01|                9900|
|     5|    Eve|Finance|New York|  5100|2022-03-05|               16100|
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|               18300|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|               13200|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|                6500|
|     1|  Alice|     HR|New York|  3500|2022-01-10|               10400|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|               11600|
|     2|    Bob|     HR|New York|  4200|2022-01-15|                8100|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|               11500|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|               18700|
|     4|  David|     IT| Chicago|  7200|2022-02-10|

In [18]:
df.withColumn("Running Salary Total", sum("salary").over(win.rowsBetween(-1, Window.unboundedFollowing))).show()

+------+-------+-------+--------+------+----------+--------------------+
|emp_id|   name|   dept|location|salary| hire_date|Running Salary Total|
+------+-------+-------+--------+------+----------+--------------------+
|    11|    Ken|Finance|New York|  4800|2022-06-01|               23100|
|     5|    Eve|Finance|New York|  5100|2022-03-05|               23100|
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|               18300|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|               13200|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|               14600|
|     1|  Alice|     HR|New York|  3500|2022-01-10|               14600|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|               11600|
|     2|    Bob|     HR|New York|  4200|2022-01-15|                8100|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|               26700|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|               26700|
|     4|  David|     IT| Chicago|  7200|2022-02-10|

#### rangeBetween

In [19]:
df.show()

+------+-------+-------+--------+------+----------+
|emp_id|   name|   dept|location|salary| hire_date|
+------+-------+-------+--------+------+----------+
|     1|  Alice|     HR|New York|  3500|2022-01-10|
|     2|    Bob|     HR|New York|  4200|2022-01-15|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|
|     4|  David|     IT| Chicago|  7200|2022-02-10|
|     5|    Eve|Finance|New York|  5100|2022-03-05|
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|
|     7|  Grace|     IT|  Boston|  5500|2022-04-01|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|
|    10|   Jack|     IT|  Boston|  8000|2022-05-15|
|    11|    Ken|Finance|New York|  4800|2022-06-01|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|
+------+-------+-------+--------+------+----------+



In [20]:
win = Window.partitionBy("dept").orderBy(col("salary").desc())

In [21]:
df.withColumn("Running Total Salary", sum("salary").over(win.rangeBetween(-1000, 1000))).show()

+------+-------+-------+--------+------+----------+--------------------+
|emp_id|   name|   dept|location|salary| hire_date|Running Total Salary|
+------+-------+-------+--------+------+----------+--------------------+
|     9|    Ivy|Finance| Chicago|  7000|2022-05-02|               13200|
|     6|  Frank|Finance|  Boston|  6200|2022-03-20|               13200|
|     5|    Eve|Finance|New York|  5100|2022-03-05|                9900|
|    11|    Ken|Finance|New York|  4800|2022-06-01|                9900|
|     2|    Bob|     HR|New York|  4200|2022-01-15|               11600|
|    12|   Lily|     HR| Chicago|  3900|2022-06-10|               14600|
|     1|  Alice|     HR|New York|  3500|2022-01-10|               14600|
|     8|   Hank|     HR|  Boston|  3000|2022-04-12|               10400|
|    10|   Jack|     IT|  Boston|  8000|2022-05-15|               15200|
|     4|  David|     IT| Chicago|  7200|2022-02-10|               15200|
|     3|Charlie|     IT| Chicago|  6000|2022-02-01|

In [63]:
spark.stop()